## Elevator

A worked example of `FormalVizWidget` built from
`spec/ElevatorSpectabular.ipynb`'

The SVG is adapted from `report/report.ipynb` section 11's Anywidget prototype
(`ElevatorFormalWidget`), reduced from 5 to 3 floors and adapted to the
real spec's variables (`floor`, `dir`, `b1u`/`b2u`/`b2d`/`b3d`,
`c1`/`c2`/`c3`). The door from that prototype is dropped for now.

Pull the framework notebook into this kernel first:

In [1]:
%run "../FormalVizWidget.ipynb"

### State & rendering

`state` is the spec's own variables: which floor the cabin is at, its
travel direction, the four exterior call buttons, and the three cabin
buttons. `ANIMATION_HANDLER` moves the cabin to the new floor's height,
toggles each button indicator, and updates the direction/request text.

In [2]:
INITIAL_STATE = {
    "floor": "F1",
    "dir": "Idle",
    "b1u": False,
    "b2u": False,
    "b2d": False,
    "b3d": False,
    "c1": False,
    "c2": False,
    "c3": False,
}

ANIMATION_HANDLER = """
    // prev, next = state dicts; svg = SVG element;
    // duration = ms; tween(el, {cssProp: val}, ms); snap(el, {prop: val})

    const FLOOR_Y = { F1: 130, F2: 80, F3: 30 };
    const BTN_ON = "#ff9800";
    const BTN_OFF = "#bdbdbd";
    const EXT_BUTTONS = ["b1u", "b2u", "b2d", "b3d"];
    const CAB_BUTTONS = ["c1", "c2", "c3"];
    const DIR_LABEL = { Up: "\u25b2 up", Down: "\u25bc down", Idle: "\u25cf idle" };

    if (prev.floor !== next.floor) {
        const dy = FLOOR_Y[next.floor] - FLOOR_Y["F1"];
        tween(svg.getElementById("cabin-group"), { transform: "translateY(" + dy + "px)" }, duration);
    }

    EXT_BUTTONS.concat(CAB_BUTTONS).forEach(name => {
        if (prev[name] !== next[name]) {
            tween(svg.getElementById("btn-" + name), { fill: next[name] ? BTN_ON : BTN_OFF }, duration);
        }
    });

    // texts
    svg.getElementById("floor-display").textContent = next.floor;
    svg.getElementById("dir-display").textContent = DIR_LABEL[next.dir] ?? next.dir;

    const activeExt = EXT_BUTTONS.filter(name => next[name]);
    svg.getElementById("req-display").textContent = "requests: " + (activeExt.length ? activeExt.join(",") : "none");

    const activeCab = CAB_BUTTONS.filter(name => next[name]);
    svg.getElementById("cab-display").textContent = "cabin: " + (activeCab.length ? activeCab.join(",") : "none");
"""


def create_elevator_widget(svg_path="Elevator.svg", viewbox=None):
    return ElevatorWidget.from_svg(
        svg_path,
        variables=INITIAL_STATE,
        animation_handler=ANIMATION_HANDLER,
        viewbox=viewbox,
    )

### Spectabular spec:
`ElevatorSpectabular.ipynb`'s own first cell does
`%run "spectabular.ipynb"`, which is a path relative to *its own* directory
(`spec/`). A nested `%run` doesn't change the kernel's working directory,
so invoking it directly from here (cwd `anywidget/elevator/`) would still
look for `spectabular.ipynb` beside this notebook and fail. To fix this,`chdir`
into `spec/` for the duration of the `%run`, then back.

In [3]:
import os

here = os.getcwd()
os.chdir("../../spec")
get_ipython().run_line_magic("run", '"ElevatorSpectabular.ipynb"')
os.chdir(here)

### `ElevatorWidget`

In [4]:
elevatorSpec = buttonEvent | arrivalEvent


class ElevatorWidget(FormalVizWidget):
    spec = elevatorSpec

### Usage

Each cell below is independently runnable once the cells above have
executed.

Create the widget:

In [5]:
w = create_elevator_widget(viewbox="0 0 380 220")
w

In [6]:
#w.compute_next_state(w.state, ev = "Press3D")
#w.spec

Press the floor-2 up button, then arrive at F2 (which clears it and stops, since nothing else is pending):

In [7]:
w.reset()
next_state = w.compute_next_state(w.state, ev="Press2U")
await w.transition(next_state, duration=0.3)
next_state = w.compute_next_state(w.state, ev="ArrF2")
await w.transition(next_state, duration=0.6)

A passenger inside presses cabin button 3 (`PressC3`), and someone outside calls for floor 2 going up (`Press2U`): the elevator heads up and, following the elevator algorithm, continues past floor 2 to floor 3 first, since a request remains above:

In [8]:
await reset_and_run(w, run_scenario, [
    {"ev": "PressC3"},
    {"ev": "Press2U"},
    {"ev": "ArrF2"},   # continues Up: c3 still pending above
    {"ev": "ArrF3"},   # no requests above F3 now: stops
], transition_dur=0.6)

Press the floor-2 *down* button while idle below it: the elevator must first travel up to fetch the passenger, then reverses direction immediately on arrival, since the only pending request is behind it, and continues back down:

In [9]:
await reset_and_run(w, run_scenario, [
    {"ev": "Press2D"},
    {"ev": "ArrF2"},   # reqAboveF2 false, b2d true: reverses to Down
    {"ev": "ArrF1"},
], transition_dur=0.6)